# Saltwater intrusion in the advanced coastal valley

The advanced synthetic valley routes its stream with streamflow routing
(**SFR**), holds its lake in a lake (**LAK**) package, moves recharge through
the unsaturated zone (**UZF**), pumps from multi-aquifer wells (**MAW**),
collects seepage with drains (**DRN**), and routes water between them with the
water mover (**MVR**). Put a sea along its southern edge and the same saltwater
wedge forms as in the base valley, but now the stream and the wells are the
freshwater features the salt can reach.

## What this notebook covers

Couple flow and transport on the advanced coastal valley, let the Buoyancy
(**BUY**) package make the seawater heavier than the fresh water, and follow
where the salt goes.

By the end of this notebook you will be able to:

- couple a transport model to a flow model that uses the advanced packages,
- say why that needs a water mover transport (**MVT**) package but not SFT,
  LKT, or UZT,
- read the salt that leaves through the stream and the wells from the transport
  budget, and
- compare the wedge under the advanced valley with the one under the base
  valley.

The base valley is covered in
[`mf6-coastal-density`](mf6-coastal-density.ipynb), and the coastal boundary
itself in [`mf6-coastal-ghb`](mf6-coastal-ghb.ipynb).

Import the packages this notebook uses. `mf6_coastal_density` holds the model
builder and the readers, so the cells below stay about the hydrology. The models
are in meters and days; set the environment variable `MF6_TRAINING_UNITS` to
`feet` to see lengths, heads, and flows in feet.

In [ ]:
%matplotlib inline

import pathlib as pl

import flopy
import matplotlib.pyplot as plt
import mf6_coastal_density as cd
import numpy as np
import pandas as pd
from flopy.utils.postprocessing import get_specific_discharge
from mf6_notebook_helpers import find_mf6_libraries, length_name, to_display, unit

_, mf6_exe = find_mf6_libraries()

## How density enters the flow equation

MODFLOW 6 solves for hydraulic head, and the BUY package turns a simulated
concentration into a fluid density with a straight line:

```
density = denseref + drhodc * concentration
```

`denseref` is the density of fresh water and `drhodc` is the slope. The values
below put seawater at 1,024.5 kg/m³ against 1,000 for fresh water. Only the
ratio of the two enters the flow equation, so the concentration unit does not
have to match the length unit of the model. Here it does: with salinity in
kg/m³ and the valley in meters, the transport budget is in kilograms of salt per
day.

The dispersivities, the length over which the salt and fresh water mix, are a
tenth of the 152.4-m (500-ft) cell along the direction of flow and a hundredth
and a thousandth of it across the flow horizontally and vertically.

In [ ]:
print(f"fresh water:  {cd.DENSEREF:8.1f} kg/m3")
print(f"seawater:     {cd.DENSEREF + cd.DRHODC * cd.SEAWATER:8.1f} kg/m3")
print(f"density ratio: {(cd.DENSEREF + cd.DRHODC * cd.SEAWATER) / cd.DENSEREF:7.4f}")
print(f"porosity:     {cd.POROSITY:8.2f}")
print(
    f"dispersivity: {to_display(cd.ALH):8.2f} {unit(plain=True)} along flow, "
    f"{to_display(cd.ATH1):.2f} {unit(plain=True)} across it horizontally, "
    f"{to_display(cd.ATV):.3f} {unit(plain=True)} vertically"
)

## Coupling transport to the advanced packages

Every stress package a flow model carries has to tell the transport model what
concentration its water arrives at. For ordinary packages the Source and Sink
Mixing (**SSM**) package does that job, and it does the same for SFR, LAK, UZF,
MAW, and the drains as long as no advanced transport package has claimed them.
Water entering the aquifer from the stream, the lake, or the unsaturated zone
arrives fresh, and water leaving takes the salinity of the cell it leaves from.

The mover is the one package that has to be named. MODFLOW 6 stops with

```
GWF water mover is active but the GWT MVT package has not been specified.
```

unless a **MVT** package exists, so the builder adds one with no options. The
mover then carries water - runoff from the unsaturated zone and the water the
seepage drains collect - to the lake and the stream without carrying solute.
That is exact as long as the water it routes is fresh, which the transport
budget below confirms: no salt leaves through the drains or the unsaturated
zone.

What this leaves out is in-stream and in-lake transport: the stream and the lake
do not hold a salinity of their own, so salt that reaches them leaves the model
rather than travelling down the channel. Simulating that needs the streamflow
transport (**SFT**), lake transport (**LKT**), and unsaturated-zone transport
(**UZT**) packages, one per advanced package, and their own concentration data.

## Build and run the coupled model

`build_simulation()` loads the shipped coastal advanced model, shortens it to a
steady spin-up plus five pumping years, and adds the transport model, the
GWF-GWT exchange, and BUY. The shipped advanced valley leaves its prediction
well off, so the builder pumps it at the base valley's rate, 8,495 m³/d
(300,000 ft³/d), through the five years. The spin-up lets the wedge reach the
position the freshwater discharge holds it at, and the pumping years then move
it.

In [ ]:
ws = pl.Path("models/coastal-density-advanced")
sim = cd.build_simulation(ws, mf6_exe, variant="advanced")
success, buff = sim.run_simulation(silent=True)
if not success:
    raise RuntimeError("\n".join(buff[-20:]))

gwf = sim.get_model("sv")
print(f"stress periods: {sim.tdis.nper.data}")
print(f"spin-up steps:  {cd.SPINUP_NSTP}, steps per year after it: {cd.ANNUAL_NSTP}")

## The wedge

Cut a section along the column where the wedge reaches farthest inland in the
bottom layer, and plot salinity at the end of the spin-up and after pumping
years 1, 3, and 5. Each panel shows the southern 16 rows; the coast is at the
right, at row 40.

In [ ]:
conc = cd.concentration(ws)
nlay, nrow, ncol = gwf.dis.nlay.data, gwf.dis.nrow.data, gwf.dis.ncol.data
delc = float(np.atleast_1d(gwf.dis.delc.array)[0])

# the column where the wedge reaches farthest inland in the bottom layer
reach = [cd.wedge_toe(np.nan_to_num(conc[-1]), j, nlay - 1, delc) for j in range(ncol)]
section_column = int(np.argmax(reach))  # zero-based
print(f"section along column {section_column + 1}")

times = [0, 1, 3, 5]  # spin-up, then years 1, 3, and 5
with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["A", "B"], ["C", "D"]], figsize=(9.5, 6.5), layout="constrained"
    )
    for letter, itime in zip("ABCD", times):
        ax = axd[letter]
        xs = flopy.plot.PlotCrossSection(
            model=gwf, ax=ax, line={"column": section_column}
        )
        cb = xs.plot_array(conc[itime], cmap="viridis", vmin=0.0, vmax=cd.SEAWATER)
        xs.plot_grid(lw=0.2, color="0.8")
        cs = xs.contour_array(
            conc[itime], levels=[0.5 * cd.SEAWATER], colors="w", linewidths=1.5
        )
        label = "end of spin-up" if itime == 0 else f"year {itime}"
        ax.set_title(f"{letter}. {label}")
        ax.set_xlabel(f"distance from the north end, in {length_name()}")
        ax.set_ylabel(f"elevation, in {length_name()}")
        # the southern 16 rows, where the coast is
        ax.set_xlim((nrow - 16) * delc, nrow * delc)
        cd.display_ticks(ax.xaxis)
        cd.display_ticks(ax.yaxis)
    fig.colorbar(cb, ax=list(axd.values()), shrink=0.6, label="salinity, in kg/m3")

**What to look for.** The section is column 6, west of the stream in column 9.
It passes through the multi-aquifer well vc, which pumps from layers 4 and 5 in
row 33, seven rows from the coast. Salt water fills the deep cells at the coast
and thins inland, which is the wedge. The white line is the 50 percent
isochlor, the contour halfway between fresh water and seawater, and it is the
line usually used to say how far the wedge reaches. The wedge sits under the
fresh water rather than mixing with it, because it is denser. The short white
line at the top of the coastal cell from year 1 on is layers 1 and 2 at the
shore rising above half seawater.

## Where the sea water goes in and comes out

A coastal boundary is not simply a drain. Sum the boundary flow by layer for the
last time step: the bottom layer takes sea water in and the layers above it send
water out, which is the circulation a density model produces and a
constant-density model cannot.

In [ ]:
budget = flopy.utils.CellBudgetFile(ws / "sv.cbc")
ghb = budget.get_data(text="GHB", totim=budget.get_times()[-1])[0]
ncell = nrow * ncol
by_layer = np.zeros(nlay)
for node, q in zip(ghb["node"], ghb["q"]):
    by_layer[(node - 1) // ncell] += q
by_layer = to_display(by_layer, "flow")

flow_unit = unit("flow", plain=True)
for k, q in enumerate(by_layer):
    if q == 0.0:
        continue  # layer 3 has no thickness at the coast
    direction = "into the aquifer" if q > 0 else "out to the sea"
    print(f"layer {k + 1}: {q:12,.0f} {flow_unit} {direction}")
print(f"total:   {by_layer.sum():12,.0f} {flow_unit}")

**What to look for.** In the last time step layer 5 takes in 8,353 m³/d
(294,975 ft³/d) of sea water and layers 1, 2, and 4 send out 3,731 m³/d
(131,776 ft³/d), so the coast as a whole supplies 4,621 m³/d (163,199 ft³/d) to
the aquifer. In the base valley the coast drains the aquifer instead.

Follow the wedge and the salt through time. The toe is the farthest inland cell
in the bottom layer whose salinity is at least half that of seawater, measured
along the section column from the coastal edge of the grid. Panel B sums the
salt the aquifer stores, from the transport budget.

In [ ]:
toe = [
    to_display(cd.wedge_toe(np.nan_to_num(conc[i]), section_column, nlay - 1, delc))
    for i in range(conc.shape[0])
]
salt = pd.read_csv(ws / f"{cd.GWT_NAME}-budget.csv")
# salt going into storage is the OUT term; concentration in kg/m3 times
# flow in m3/d makes it kg/d, and summing over the time steps gives kg
stored = salt["STORAGE-AQUEOUS(MST)_OUT"] - salt["STORAGE-AQUEOUS(MST)_IN"]
salt_mass = np.cumsum(stored * np.diff(salt["time"], prepend=0.0))

with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["A", "B"]], figsize=(9.5, 3.6), layout="constrained"
    )

    ax = axd["A"]
    ax.plot(range(len(toe)), toe, "o-", color="tab:blue")
    ax.set_xlabel("stress period")
    ax.set_ylabel(f"toe position, in {length_name()} inland")
    ax.set_title("A. How far the wedge reaches")

    ax = axd["B"]
    # the run starts with no salt in the aquifer
    ax.plot(
        np.r_[0.0, salt["time"] / 365.25],
        np.r_[0.0, salt_mass / 1.0e6],
        color="tab:red",
    )
    ax.set_xlabel("time, in years")
    ax.set_ylabel("salt in the aquifer, in million kg")
    ax.set_title("B. Salt accumulating in the aquifer")

print(f"toe by period: {', '.join(f'{t:,.0f}' for t in toe)} {unit(plain=True)}")
nspin = cd.SPINUP_NSTP
print(
    f"salt in the aquifer after the spin-up: {salt_mass[nspin - 1] / 1.0e6:.1f} million kg"
)
print(
    f"salt in the aquifer after year 5:      {salt_mass.iloc[-1] / 1.0e6:.1f} million kg"
)

**What to look for.** The toe sits 762 m (2,500 ft) inland at the end of the
spin-up and in year 1, 914 m (3,000 ft) in years 2 to 4, and 1,067 m (3,500 ft)
in year 5, one row short of the well. The aquifer holds 485 million kg of salt at
the end of the spin-up and 906 million kg after the five pumping years, 87
percent more; the base valley gains a third over the same years.

## What density is responsible for

Run the same model again with the BUY package left out. Everything else is
identical, so the difference between the two runs is what the density contrast
does.

In [ ]:
ws_fresh = pl.Path(f"{ws}-no-buoyancy")
sim_fresh = cd.build_simulation(ws_fresh, mf6_exe, variant="advanced", buoyancy=False)
success, buff = sim_fresh.run_simulation(silent=True)
if not success:
    raise RuntimeError("\n".join(buff[-20:]))

head = to_display(flopy.utils.HeadFile(ws / "sv.hds").get_alldata()[-1])
head_fresh = to_display(flopy.utils.HeadFile(ws_fresh / "sv.hds").get_alldata()[-1])
conc_fresh = cd.concentration(ws_fresh)
toe_density = cd.wedge_toe(np.nan_to_num(conc[-1]), section_column, nlay - 1, delc)
toe_fresh = cd.wedge_toe(np.nan_to_num(conc_fresh[-1]), section_column, nlay - 1, delc)

length_unit = unit(plain=True)
print(
    "deepest-layer head, coastal row, with density:    "
    f"{head[-1, -1].mean():8.3f} {length_unit}"
)
print(
    "deepest-layer head, coastal row, without density: "
    f"{head_fresh[-1, -1].mean():8.3f} {length_unit}"
)
print(
    "largest head difference anywhere:                 "
    f"{np.nanmax(np.abs(head - head_fresh)):8.3f} {length_unit}"
)
print(f"toe with density:    {to_display(toe_density):8,.0f} {length_unit}")
print(f"toe without density: {to_display(toe_fresh):8,.0f} {length_unit}")
print(f"highest salinity without density: {np.nanmax(conc_fresh[-1]):.3f} kg/m3")

**What to look for.** Without buoyancy, water above half seawater forms only
next to the stream outlet: 4 cells in layers 4 and 5 of columns 8 and 10 of the
coastal row, the highest at 21.4 kg/m³. The section column stays below
4.2 kg/m³, so there is no wedge. With density, the head in layer 5 along the
coastal row averages -0.054 m (-0.177 ft), against -0.011 m (-0.035 ft) without
it, and heads differ by up to 1.15 m (3.76 ft). The density term is what draws
sea water in at depth along the whole coast and drives the circulation.

## Which way the water moves at the coast

A coast is usually drawn as a discharge zone, with fresh water rising through the
aquifer and leaving at the shore. Whether this model does that depends on
density. Plot the vertical specific discharge along the southern half of the
valley, averaged across each row and positive upward, in a shallow layer and a
deep one, for the density run, the run without it, and a third constant-density
run that carries the weight of the sea in its boundary heads.

In [ ]:
def vertical_flow(workspace, layer):
    """Mean vertical specific discharge by row in one layer, positive upward."""
    cbc = flopy.utils.CellBudgetFile(workspace / "sv.cbc")
    totim = cbc.get_times()[-1]
    head = flopy.utils.HeadFile(workspace / "sv.hds").get_data(totim=totim)
    spdis = cbc.get_data(text="DATA-SPDIS", totim=totim)[0]
    _, _, qz = get_specific_discharge(spdis, gwf, head=head)
    return to_display(np.nanmean(qz[layer], axis=1), "rate")


# what a constant-density model does when the weight of the sea is put in by
# hand, as a boundary head that rises with depth
ws_efh = pl.Path(f"{ws}-freshwater-head")
sim_efh = cd.build_simulation(
    ws_efh, mf6_exe, variant="advanced", buoyancy=False, equivalent_freshwater=True
)
success, buff = sim_efh.run_simulation(silent=True)
if not success:
    raise RuntimeError("\n".join(buff[-20:]))

runs = (
    ("with density", ws, "tab:blue"),
    ("constant density, sea level", ws_fresh, "tab:gray"),
    ("constant density, freshwater head", ws_efh, "tab:red"),
)
inland = to_display((nrow - 1 - np.arange(nrow)) * delc)
qz = {
    (label, layer): vertical_flow(workspace, layer)
    for label, workspace, _ in runs
    for layer in (1, 3)
}
# scale to the density and sea-level runs; the freshwater-head run is annotated
ylim = 1.1 * max(
    np.nanmax(np.abs(q)) for (label, _), q in qz.items() if "freshwater" not in label
)

with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["A", "B"]], figsize=(9.5, 3.8), layout="constrained"
    )
    for letter, layer in zip("AB", (1, 3)):
        ax = axd[letter]
        for label, _, color in runs:
            ax.plot(inland, qz[(label, layer)], color=color, label=label)
        ax.axhline(0.0, lw=0.8, color="k")
        ax.set_xlim(0.0, inland[0] / 2.0)
        ax.set_ylim(-ylim, ylim)
        peak = qz[("constant density, freshwater head", layer)][-1]
        if abs(peak) > ylim:
            ax.annotate(
                f"{peak:.3f} {unit('rate')} at the coast",
                xy=(0.0, np.sign(peak) * ylim),
                xytext=(0.12 * inland[0] / 2.0, np.sign(peak) * 0.7 * ylim),
                color="tab:red",
                fontsize=7,
                arrowprops={"arrowstyle": "->", "color": "tab:red", "lw": 0.8},
            )
        ax.set_xlabel(f"distance inland from the coast, in {length_name()}")
        ax.set_ylabel(f"vertical specific discharge, in {unit('rate')}")
        ax.set_title(f"{letter}. Layer {layer + 1}")
    axd["A"].legend(fontsize=7, loc="lower right")

rate_unit = unit("rate", plain=True)
for (label, layer), q in qz.items():
    print(f"layer {layer + 1}, {label}: {q[-1]:.4f} {rate_unit} at the coast")

**What to look for.** As in the base valley, the three runs disagree about which
way the water moves at the shore.

With a sea-level boundary and no density the water moves down all the way from
1,524 m (5,000 ft) inland to the coast, at -0.0001 m/d (-0.0003 ft/d) in layer 4
at the shore. The upward band that ends 2,591 m (8,500 ft) inland, at the head
of the stream, is in all three runs and belongs to the stream. The dip 762 m
(2,500 ft) inland in layer 4, also in all three, is the prediction well pulling
water down into layer 5.

With density the near-shore flow reverses and layer 4 rises at 0.0038 m/d
(0.0124 ft/d) at the coast. That is the circulation in profile: sea water
flowing in at depth, mixing, and leaving as brackish water through the layers
above.

The equivalent freshwater head again overshoots. It drives layer 4 upward at
0.0531 m/d (0.1743 ft/d) at the coast, fourteen times the density run, and keeps
the flow upward for the first 610 m (2,000 ft) inland in both layers. It gets
the direction right and the magnitude wrong.

The freshwater-head run runs off the top of both panels; its value at the coast
is annotated on each.

## Where the salt leaves

The transport budget has one entry per flow package, so it says which features
the salt leaves through. Average it over the last pumping year, whose twelve time
steps are equal in length.

In [ ]:
# average over the 12 time steps of the last year, which are equal in length
last_year = salt[salt["time"] > salt["time"].iloc[-1] - 365.0].mean()
for column in salt.columns:
    if "SSM" in column and column.endswith("_OUT") and last_year[column] >= 1.0:
        print(f"{column:28s} {last_year[column]:12,.0f} kg/d")

**What to look for.** Salt leaves through the coast, the stream, and the wells.
The most, 91,973 kg/d, goes back to the sea through the upper coastal cells; the
stream carries away 55,601 kg/d, the multi-aquifer wells pump 45,440 kg/d, and
the prediction well 25,634 kg/d. Pumping near a coast draws salt toward the
wells, and a stream in contact with a salty aquifer carries that salt away.
Nothing leaves through the lake, the drains, or the unsaturated zone.

## Recap

- The BUY package turns simulated salinity into fluid density with a reference
  density and a slope, and the flow and transport models are solved together
  through a GWF-GWT exchange.
- Sea water enters the aquifer through the deep coastal cells and water leaves
  through the shallow ones, which is the circulation that a constant-density
  model cannot produce.
- The 50 percent isochlor is the usual measure of how far the wedge reaches.
- Running the same model without buoyancy separates what the density contrast
  does from what the boundary does.
- SSM handles SFR, LAK, UZF, MAW, and DRN as ordinary boundaries, so no SFT,
  LKT, or UZT package is needed; the mover needs an MVT package, which can be
  empty when everything the mover routes is fresh.